## Desigualdades de Gênero na Ciência de Dados (2021–2024)

**Estratégia:**
- **Treino:** anos 2021, 2022 e 2023 (dados sem perguntas de IA)
- **Teste / drift:** 2024 (inclui colunas de adoção de IA generativa)
- **Targets:** Faixa Salarial · Nível de Senioridade · Cargo de Liderança
- **Modelos:** Regressão Logística · Random Forest · XGBoost
- **Explicabilidade:** SHAP

## 1. Imports e configurações

In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (classification_report, accuracy_score, f1_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)
os.makedirs('figures', exist_ok=True)

print("✅ Imports OK")

✅ Imports OK


In [10]:
df_full = pd.read_excel("df_full.xlsx")

## 5. Feature engineering e definição de targets

**Targets:**
| Target | Tipo | Observação |
|--------|------|------------|
| `faixa_salarial` | Ordinal (12 classes) | Codificada como int 0–11 |
| `nivel` | Ordinal (4 classes) | Júnior=0 … Gestor=3 |
| `gestor` | Binário | 1=sim, 0=não |

**Features base** (disponíveis em 2021–2024): formação, experiência, cargo, modalidade, região, setor, situação.

**Features de IA** (só 2024): usadas na análise exploratória e em modelo extra pós-2024.

In [11]:
# ---------------------------------------------------------------------------
# Mapeamentos de categorias
# ---------------------------------------------------------------------------

FAIXA_SALARIAL_MAP = {
    'de R$ 1.001/mês a R$ 2.000/mês':   'R$1k-2k',
    'de R$ 2.001/mês a R$ 3.000/mês':   'R$2k-3k',
    'de R$ 2.001/mês a R$ 3000/mês':    'R$2k-3k',
    'de R$ 3.001/mês a R$ 4.000/mês':   'R$3k-4k',
    'de R$ 4.001/mês a R$ 6.000/mês':   'R$4k-6k',
    'de R$ 6.001/mês a R$ 8.000/mês':   'R$6k-8k',
    'de R$ 8.001/mês a R$ 12.000/mês':  'R$8k-12k',
    'de R$ 12.001/mês a R$ 16.000/mês': 'R$12k-16k',
    'de R$ 16.001/mês a R$ 20.000/mês': 'R$16k-20k',
    'de R$ 20.001/mês a R$ 25.000/mês': 'R$20k-25k',
    'de R$ 25.001/mês a R$ 30.000/mês': 'R$25k-30k',
    'de R$ 30.001/mês a R$ 40.000/mês': 'R$30k-40k',
    'Acima de R$ 40.001/mês':           'R$40k+',
}
FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k','R$2k-3k','R$3k-4k','R$4k-6k','R$6k-8k',
    'R$8k-12k','R$12k-16k','R$16k-20k','R$20k-25k',
    'R$25k-30k','R$30k-40k','R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal', 'Estudante de Graduação',
    'Graduação/Bacharelado', 'Especialização Lato Sensu',
    'Mestrado', 'Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior', 'Gestor']
CARGO_MAP = {
    'Analista de Dados/Data Analyst':                    'Analista de Dados',
    'Analista de BI/BI Analyst':                         'Analista de Dados',
    'Analista de BI/BI Analyst/Analytics Engineer':      'Analista de Dados',
    'Analista de Negócios/Business Analyst':             'Analista de Dados',
    'Analista de Inteligência de Mercado/Market Intelligence': 'Analista de Dados',
    'Analista de Marketing':                             'Analista de Dados',
    'Analista Administrativo':                           'Analista de Dados',
    'Estatístico':                                       'Analista de Dados',
    'Economista':                                        'Analista de Dados',
    'Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect': 'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer':                 'Engenheiro de Dados',
    'Analytics Engineer':                                'Engenheiro de Dados',
    'Arquiteto de Dados':                                'Engenheiro de Dados',
    'Arquiteto de dados':                                'Engenheiro de Dados',
    'DBA/Administrador de Banco de Dados':               'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer/Data Architect':  'Engenheiro de Dados',
    'Arquiteto de Dados/Data Architect':                 'Engenheiro de Dados',
    'Cientista de Dados/Data Scientist':                 'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer':        'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer/AI Engineer': 'Cientista de Dados',
    'Professor':                                         'Professor/Pesquisador',
    'Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Product Manager/ Product Owner (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas': 'Desenvolvedor',
    'Desenvolvedor ou Engenheiro de Software':           'Desenvolvedor',
    'Analista de Sistemas/Analista de TI':               'Desenvolvedor',
    'Analista de Suporte/Analista Técnico':              'Desenvolvedor',
    'Suporte Técnico':                                   'Desenvolvedor',
    'Técnico':                                           'Desenvolvedor',
    'Outra Opção':                                       'Outro',
    'Outras Engenharias (não inclui dev)':               'Outro',
}
AREA_FORMACAO_MAP = {
    'Computação / Engenharia de Software / Sistemas de Informação/ TI': 'Computação / Engenharia de Software / TI',
    'Outras Engenharias':                                'Engenharia (outras)',
    'Outras Engenharias (não incluir engenharia de software ou TI)': 'Engenharia (outras)',
    'Economia/ Administração / Contabilidade / Finanças/ Negócios': 'Economia / Administração / Finanças / Negócios',
    'Economia/ Administração / Contabilidade / Finanças': 'Economia / Administração / Finanças / Negócios',
    'Estatística/ Matemática / Matemática Computacional/ Ciências Atuariais': 'Estatística / Matemática / Ciências Atuariais',
    'Estatística/ Matemática / Matemática Computacional': 'Estatística / Matemática / Ciências Atuariais',
    'Ciências Biológicas/ Farmácia/ Medicina/ Área da Saúde': 'Ciências Biológicas / Medicina / Saúde',
    'Ciências Biológicas/Farmácia/Medicina/Área da Saúde': 'Ciências Biológicas / Medicina / Saúde',
    'Marketing / Publicidade / Comunicação / Jornalismo': 'Marketing / Comunicação / Jornalismo',
    'Marketing / Publicidade / Comunicação / Jornalismo / Ciências Sociais': 'Marketing / Comunicação / Jornalismo',
    'Outra opção':                                       'Outras',
}
SITUACAO_MAP = {
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil':       'Remoto para fora do país',
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil (PJ)':  'Remoto para fora do país',
    'Desempregado, buscando recolocação':                                    'Desempregado',
    'Desempregado e não estou buscando recolocação':                         'Desempregado',
    'Somente Estudante (graduação)':                                         'Somente Estudante',
    'Somente Estudante (pós-graduação)':                                     'Somente Estudante',
    'Prefiro não informar':                                                   None,
}
IA_USO_PESSOAL_MAP = {
    'Não uso soluções de AI Generativa com foco em produtividade':               'Não usa',
    'Uso soluções gratuitas de AI Generativa com foco em produtividade':         'Usa (gratuito)',
    'Uso e pago pelas soluções de AI Generativa com foco em produtividade':      'Usa (pago próprio)',
    'A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade': 'Usa (pago empresa)',
    'Uso soluções do tipo Copilot':                                              'Usa Copilot',
}

print("✅ Mapeamentos definidos")


✅ Mapeamentos definidos


In [12]:
# Features base comuns a todos os anos
BASE_FEATURES = [
    'nivel_ensino',     # ordinal → codificada como int
    'area_formacao',    # nominal
    'cargo',            # nominal  
    'tempo_area_dados', # nominal (faixas)
    'tempo_area_ti',    # nominal (faixas)
    'modalidade',       # nominal
    'setor',            # nominal
    'regiao',           # nominal
    'situacao',         # nominal
]

# Colunas de IA disponíveis no df_full
IA_FEATURES = [c for c in df_full.columns if c.startswith('ia_')]
print(f"Colunas de IA disponíveis: {len(IA_FEATURES)}")
print(IA_FEATURES)


def encode_ordinal(series, ordem):
    """Codifica variável ordinal como int. Categorias não mapeadas → NaN."""
    cat = pd.Categorical(series, categories=ordem, ordered=True)
    codes = cat.codes.astype(float)
    codes[codes == -1] = np.nan
    return codes


def prepare_xy(df: pd.DataFrame, target: str, extra_features: list = None):
    """
    Prepara X e y para um dado target.
    - nivel_ensino é codificada como int (ordinal).
    - Demais features permanecem como str/object para OHE no pipeline.
    - Retorna também metadados (genero, ano) para análise de equidade.
    """
    feats = BASE_FEATURES.copy()
    if extra_features:
        feats += [f for f in extra_features if f in df.columns]

    sub = df[feats + [target, 'ano_pesquisa', 'genero']].copy()

    # Ordinal encoding de nivel_ensino
    sub['nivel_ensino'] = encode_ordinal(sub['nivel_ensino'], NIVEL_ENSINO_ORDEM)

    # Target
    if target == 'faixa_salarial':
        sub['y'] = encode_ordinal(sub['faixa_salarial'], FAIXA_SALARIAL_ORDEM)
    elif target == 'nivel':
        sub['y'] = encode_ordinal(sub['nivel'], NIVEL_SENIORIDADE_ORDEM)
    elif target == 'gestor':
        sub['y'] = (sub['gestor'] == 1.0).astype(float)

    sub = sub.dropna(subset=['y'])
    sub['y'] = sub['y'].astype(int)

    X = sub[feats].copy()
    y = sub['y']
    meta = sub[['genero', 'ano_pesquisa']]
    return X, y, meta


print("✅ Feature engineering definido")


Colunas de IA disponíveis: 0
[]
✅ Feature engineering definido


## 6. Pipeline de ML

O `ColumnTransformer` detecta automaticamente colunas numéricas e categóricas:
- **Numéricas** → imputação por mediana + StandardScaler
- **Categóricas** → imputação por moda + OneHotEncoder (handle_unknown='ignore')

In [13]:
def build_preprocessor(X: pd.DataFrame):
    """ColumnTransformer automático baseado nos dtypes de X."""
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    transformers = []
    if num_cols: transformers.append(('num', num_pipe, num_cols))
    if cat_cols: transformers.append(('cat', cat_pipe, cat_cols))
    return ColumnTransformer(transformers, remainder='drop'), num_cols, cat_cols


def get_models(task_type: str = 'multiclass') -> dict:
    """Retorna os 3 modelos com hiperparâmetros base."""
    if task_type == 'binario':
        return {
            'Regressão Logística': LogisticRegression(
                max_iter=500, random_state=SEED, class_weight='balanced'),
            'Random Forest': RandomForestClassifier(
                n_estimators=200, random_state=SEED, class_weight='balanced', n_jobs=-1),
            'XGBoost': XGBClassifier(
                n_estimators=200, random_state=SEED, eval_metric='logloss',
                scale_pos_weight=5, verbosity=0),
        }
    else:
        return {
            'Regressão Logística': LogisticRegression(
                max_iter=500, random_state=SEED, class_weight='balanced',
                multi_class='multinomial'),
            'Random Forest': RandomForestClassifier(
                n_estimators=200, random_state=SEED, class_weight='balanced', n_jobs=-1),
            'XGBoost': XGBClassifier(
                n_estimators=200, random_state=SEED, eval_metric='mlogloss', verbosity=0),
        }


def run_cv(X, y, models: dict, cv: int = 5) -> pd.DataFrame:
    """Cross-validation estratificado, métrica F1-macro."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    results = []
    for name, model in models.items():
        pre, _, _ = build_preprocessor(X)
        pipe = Pipeline([('pre', pre), ('clf', model)])
        scores = cross_val_score(pipe, X, y, cv=skf, scoring='f1_macro', n_jobs=-1)
        results.append({'Modelo': name, 'F1_macro_mean': scores.mean(), 'F1_macro_std': scores.std()})
        print(f"    {name:25s}: {scores.mean():.3f} ± {scores.std():.3f}")
    return pd.DataFrame(results)


print("✅ Pipeline definido")


✅ Pipeline definido


## 7. Treino (2021–2023) e Teste (2024)

Para cada target:
1. Cross-validation 5-fold no conjunto de treino → seleciona melhor modelo por F1-macro
2. Retreina o melhor modelo no treino completo
3. Avalia em 2024
4. Quebra o desempenho por gênero (análise de equidade / model drift)

In [14]:
def train_and_evaluate(target: str, task_type: str,
                       df_train: pd.DataFrame, df_test: pd.DataFrame):
    """
    Treina em 2021-2023, avalia em 2024.
    Retorna: pipeline treinado, nome do melhor modelo, dados de avaliação.
    """
    print(f"\n{'='*60}")
    print(f"  TARGET: {target.upper()}")
    print(f"{'='*60}")

    X_train, y_train, meta_train = prepare_xy(df_train, target)
    X_test,  y_test,  meta_test  = prepare_xy(df_test,  target)

    print(f"  Treino: {len(X_train):,} | Teste (2024): {len(X_test):,}")
    print(f"  Classes: {sorted(y_train.unique())}")

    # --- CV ---
    print("\n  [Cross-Validation 5-fold — treino 2021-2023]")
    models   = get_models(task_type)
    cv_df    = run_cv(X_train, y_train, models)
    best_name = cv_df.loc[cv_df['F1_macro_mean'].idxmax(), 'Modelo']
    print(f"\n  🏆 Melhor modelo: {best_name}")

    # Treina no full train
    pre, _, _ = build_preprocessor(X_train)
    best_pipe = Pipeline([('pre', pre), ('clf', models[best_name])])
    best_pipe.fit(X_train, y_train)

    # --- Avalia em 2024 ---
    y_pred = best_pipe.predict(X_test)
    print(f"\n  [Relatório de classificação — 2024]")
    print(classification_report(y_test, y_pred, zero_division=0))

    # --- Equidade por gênero ---
    gender_res = []
    for g in ['Masculino', 'Feminino']:
        mask = meta_test['genero'] == g
        if mask.sum() > 0:
            gender_res.append({
                'Gênero': g, 'N': int(mask.sum()),
                'Acurácia': accuracy_score(y_test[mask], y_pred[mask]),
                'F1_macro': f1_score(y_test[mask], y_pred[mask], average='macro', zero_division=0),
            })
    print("\n  [Desempenho por Gênero — 2024]")
    print(pd.DataFrame(gender_res).to_string(index=False))

    return best_pipe, best_name, X_train, X_test, y_test, y_pred, cv_df, gender_res, meta_test


# Divide treino/teste
df_train = df_full[df_full['ano_pesquisa'].isin(['2021', '2022', '2023'])].copy()
df_test  = df_full[df_full['ano_pesquisa'] == '2024'].copy()

# ─── Roda os 3 targets ──────────────────────────────────────────────────────
results = {}
for target, task_type in [('faixa_salarial', 'multiclass'),
                           ('nivel',          'multiclass'),
                           ('gestor',         'binario')]:
    out = train_and_evaluate(target, task_type, df_train, df_test)
    results[target] = {
        'pipe': out[0], 'best_name': out[1],
        'X_train': out[2], 'X_test': out[3],
        'y_test': out[4], 'y_pred': out[5],
        'cv_df': out[6], 'gender_res': out[7], 'meta_test': out[8],
    }



  TARGET: FAIXA_SALARIAL
  Treino: 0 | Teste (2024): 0
  Classes: []

  [Cross-Validation 5-fold — treino 2021-2023]


TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

## 8. Análise exploratória — Adoção de IA por Gênero (2024)

Esta seção é exclusiva para o dataset de 2024, explorando como homens e mulheres se diferenciam no uso de ferramentas de IA generativa.


In [ ]:
df_2024 = df_full[df_full['ano_pesquisa'] == '2024'].copy()

# 8.1 — Uso pessoal de IA
print("=== Uso pessoal de IA por Gênero (%) ===")
if 'ia_uso_pessoal' in df_2024.columns:
    ct = pd.crosstab(df_2024['genero'], df_2024['ia_uso_pessoal'], normalize='index') * 100
    display(ct.round(1))

# 8.2 — Features binárias de IA
ia_bin_cols = [c for c in IA_FEATURES if c not in ('ia_uso_pessoal',)
               and df_2024[c].dropna().isin([0, 1, 0.0, 1.0]).all()]

if ia_bin_cols:
    print("\n=== Taxa de adoção (%) de features de IA por Gênero ===")
    adocao = df_2024.groupby('genero')[ia_bin_cols].mean() * 100
    display(adocao.round(1).T.sort_values('Feminino', ascending=False))

## 9. Visualizações

In [ ]:
PALETTE = {'Feminino': '#E97B99', 'Masculino': '#4C8FBF'}
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

TARGET_LABELS = {
    'faixa_salarial': 'Faixa Salarial',
    'nivel':          'Senioridade',
    'gestor':         'Gestor/a',
}

# ── 9.1 Comparação de modelos (CV) ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors_bar = ['#4C8FBF', '#E97B99', '#58A868']

for ax, (target, label) in zip(axes, TARGET_LABELS.items()):
    cv_df = results[target]['cv_df']
    bars = ax.barh(cv_df['Modelo'], cv_df['F1_macro_mean'],
                   xerr=cv_df['F1_macro_std'], color=colors_bar, capsize=4)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('F1-macro (CV 5-fold)')
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    for bar, v in zip(bars, cv_df['F1_macro_mean']):
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)

fig.suptitle('Comparação de Modelos — Treino 2021-2023 (CV 5-fold)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/01_cv_comparacao_modelos.png', bbox_inches='tight')
plt.show()

# ── 9.2 Desempenho por gênero ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, (target, label) in enumerate(TARGET_LABELS.items()):
    best = results[target]['best_name']
    for row, metric in enumerate(['Acurácia', 'F1_macro']):
        ax = axes[row][col]
        df_g = pd.DataFrame(results[target]['gender_res'])
        colors = [PALETTE.get(g, 'gray') for g in df_g['Gênero']]
        ax.bar(df_g['Gênero'], df_g[metric], color=colors, width=0.5)
        ax.set_title(f'{label} — {metric}\n({best})', fontsize=9)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        for i, v in enumerate(df_g[metric]):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Desempenho por Gênero em 2024 (model drift / equidade)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/02_gender_performance.png', bbox_inches='tight')
plt.show()

# ── 9.3 Uso de IA por gênero ────────────────────────────────────────────────
if 'ia_uso_pessoal' in df_2024.columns and df_2024['ia_uso_pessoal'].notna().sum() > 0:
    ct = pd.crosstab(df_2024['genero'], df_2024['ia_uso_pessoal'], normalize='index')
    ordem_col = [v for v in IA_USO_PESSOAL_MAP.values() if v in ct.columns]
    ct = ct[ordem_col] if ordem_col else ct

    fig, ax = plt.subplots(figsize=(11, 5))
    ct.T.plot(kind='bar', ax=ax, color=[PALETTE['Feminino'], PALETTE['Masculino']], width=0.65)
    ax.set_xlabel('')
    ax.set_ylabel('Proporção')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title('Uso pessoal de IA generativa por Gênero (2024)', fontweight='bold')
    ax.legend(title='Gênero', loc='upper right')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('figures/03_ia_uso_genero.png', bbox_inches='tight')
    plt.show()

print("✅ Figuras salvas em figures/")


## 10. Explicabilidade — SHAP

Calculamos os SHAP values do melhor modelo treinado para **Faixa Salarial** e **Senioridade**, analisando quais features mais impactam as predições e se há diferenças por gênero.

In [ ]:
def compute_shap(pipe, X: pd.DataFrame, model_name: str, max_samples: int = 500):
    """Extrai SHAP values com amostragem para eficiência."""
    pre = pipe.named_steps['pre']
    clf = pipe.named_steps['clf']
    X_t = pre.transform(X)
    
    try:
        feat_names = pre.get_feature_names_out()
    except Exception:
        feat_names = np.array([f'f{i}' for i in range(X_t.shape[1])])

    n = min(max_samples, X_t.shape[0])
    idx_s = np.random.choice(X_t.shape[0], n, replace=False)
    X_s = X_t[idx_s]

    if 'XGBoost' in model_name or 'Random Forest' in model_name:
        explainer = shap.TreeExplainer(clf)
        sv = explainer.shap_values(X_s)
    else:
        explainer = shap.LinearExplainer(clf, X_s, feature_dependence='independent')
        sv = explainer.shap_values(X_s)

    # Multiclass → média do valor absoluto entre classes
    if isinstance(sv, list):
        mean_abs = np.mean([np.abs(s) for s in sv], axis=0)
    else:
        mean_abs = np.abs(sv)

    return mean_abs.mean(axis=0), feat_names, X_s


def plot_shap_bar(mean_shap, feat_names, top_n=15, title='SHAP', fname='shap.png'):
    top_idx = np.argsort(mean_shap)[::-1][:top_n]
    names   = [str(feat_names[i]) for i in top_idx]
    # Limpa prefixos do ColumnTransformer
    names_clean = [n.replace('cat__', '').replace('num__', '').replace('_cod','') for n in names]
    vals    = mean_shap[top_idx]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(names_clean[::-1], vals[::-1], color='#4C8FBF')
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f"  💾 figures/{fname}")


# ── SHAP para Faixa Salarial ────────────────────────────────────────────────
try:
    sal = results['faixa_salarial']
    shap_sal, feat_sal, _ = compute_shap(sal['pipe'], sal['X_test'], sal['best_name'])
    plot_shap_bar(shap_sal, feat_sal, title=f"SHAP — Faixa Salarial ({sal['best_name']})",
                  fname='04_shap_faixa_salarial.png')

    print("\nTop 10 features (Faixa Salarial):")
    top_idx = np.argsort(shap_sal)[::-1][:10]
    for i in top_idx:
        fname_clean = str(feat_sal[i]).replace('cat__','').replace('num__','')
        print(f"  {fname_clean:50s} {shap_sal[i]:.4f}")
except Exception as e:
    print(f"⚠️ SHAP Faixa Salarial: {e}")

# ── SHAP para Senioridade ───────────────────────────────────────────────────
try:
    niv = results['nivel']
    shap_niv, feat_niv, _ = compute_shap(niv['pipe'], niv['X_test'], niv['best_name'])
    plot_shap_bar(shap_niv, feat_niv, title=f"SHAP — Senioridade ({niv['best_name']})",
                  fname='05_shap_senioridade.png')
except Exception as e:
    print(f"⚠️ SHAP Senioridade: {e}")


## Conclusão

In [ ]:
print("=" * 70)
print("RESUMO FINAL — MELHOR MODELO POR TARGET")
print("=" * 70)

rows = []
for target, label in TARGET_LABELS.items():
    r = results[target]
    cv_best = r['cv_df'].loc[r['cv_df']['F1_macro_mean'].idxmax()]
    gdf = pd.DataFrame(r['gender_res'])
    f1_m = gdf.loc[gdf['Gênero']=='Masculino', 'F1_macro'].values
    f1_f = gdf.loc[gdf['Gênero']=='Feminino',  'F1_macro'].values
    rows.append({
        'Target':          label,
        'Melhor Modelo':   r['best_name'],
        'F1_macro CV':     f"{cv_best['F1_macro_mean']:.3f} ± {cv_best['F1_macro_std']:.3f}",
        'F1 Masc (2024)':  f"{f1_m[0]:.3f}" if len(f1_m) else '—',
        'F1 Fem (2024)':   f"{f1_f[0]:.3f}" if len(f1_f) else '—',
        'Δ (M-F)':         f"{f1_m[0]-f1_f[0]:+.3f}" if len(f1_m) and len(f1_f) else '—',
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
display(summary)
